# Page-classifier runtime benchmark

This notebook benchmarks the frozen V2 page-classification pipeline on original PDF files. It renders every selected page at 90 DPI, extracts the nine visual features, and predicts `text` or `drawing`.

The benchmark reconstructs V2 from the saved Stage 10 feature table so that training time and PDF-processing time are reported separately. It does not run cross-validation or modify the original segmentation outputs.

## 1. Configuration

Use `sample` for a 100-PDF estimate or `full` for all PDFs in `sample_data`.

In [ ]:
from pathlib import Path
from time import perf_counter
import json
import sys

import numpy as np
import pandas as pd

RUN_MODE = 'sample'
SAMPLE_PDF_COUNT = 100
RANDOM_SEED = 42

def find_classifier_dir(start: Path) -> Path:
    start = start.resolve()
    for parent in (start, *start.parents):
        for candidate in (parent, parent / 'page_classifier', parent / 'page classifier'):
            if (candidate / 'page_classifier.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate the page-classifier folder.')

CLASSIFIER_DIR = find_classifier_dir(Path.cwd())
PDF_INPUT_DIR = CLASSIFIER_DIR / 'input_pdfs'
TRAINING_CSV_PATH = CLASSIFIER_DIR / 'enriched_training_cross_validation.csv'
OUTPUT_DIR = CLASSIFIER_DIR / 'benchmark_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if RUN_MODE not in {'sample', 'full'}:
    raise ValueError("RUN_MODE must be 'sample' or 'full'.")
if not TRAINING_CSV_PATH.exists():
    raise FileNotFoundError(TRAINING_CSV_PATH)
if not PDF_INPUT_DIR.is_dir():
    raise FileNotFoundError(PDF_INPUT_DIR)

sys.path.insert(0, str(CLASSIFIER_DIR))
from page_classifier import build_model_bundle, classify_pdf

print('Run mode:', RUN_MODE)
print('PDF input:', PDF_INPUT_DIR)
print('Training table:', TRAINING_CSV_PATH)
print('Output:', OUTPUT_DIR)

## 2. Reconstruct V2 and select the benchmark PDFs

This is the only `fit()` operation in the notebook. It uses the already saved 181-row feature table and normally takes only a fraction of a second.

In [ ]:
training_start = perf_counter()
model_bundle = build_model_bundle(TRAINING_CSV_PATH)
training_seconds = perf_counter() - training_start

all_pdf_paths = sorted(PDF_INPUT_DIR.glob('*.pdf'))
if not all_pdf_paths:
    raise FileNotFoundError(f'No PDF files found in {PDF_INPUT_DIR}')

if RUN_MODE == 'full':
    benchmark_pdf_paths = all_pdf_paths
else:
    rng = np.random.default_rng(RANDOM_SEED)
    sample_size = min(SAMPLE_PDF_COUNT, len(all_pdf_paths))
    selected = np.sort(rng.choice(len(all_pdf_paths), size=sample_size, replace=False))
    benchmark_pdf_paths = [all_pdf_paths[int(index)] for index in selected]

print(f'Model reconstruction: {training_seconds:.3f} seconds')
print('Selected PDFs:', len(benchmark_pdf_paths))

## 3. Run the timed PDF benchmark

In [ ]:
prediction_frames = []
timing_rows = []
processing_start = perf_counter()

for index, pdf_path in enumerate(benchmark_pdf_paths, start=1):
    pdf_start = perf_counter()
    frame = classify_pdf(pdf_path, model_bundle)
    elapsed = perf_counter() - pdf_start
    prediction_frames.append(frame)
    timing_rows.append({
        'pdf': pdf_path.name,
        'pages': int(len(frame)),
        'seconds': elapsed,
        'seconds_per_page': elapsed / max(len(frame), 1),
    })
    if index % 10 == 0 or index == len(benchmark_pdf_paths):
        print(f'Processed {index}/{len(benchmark_pdf_paths)} PDFs')

processing_seconds = perf_counter() - processing_start
predictions = pd.concat(prediction_frames, ignore_index=True)
pdf_timings = pd.DataFrame(timing_rows)

predictions_path = OUTPUT_DIR / f'{RUN_MODE}_predictions.csv'
timings_path = OUTPUT_DIR / f'{RUN_MODE}_pdf_timings.csv'
predictions.to_csv(predictions_path, index=False)
pdf_timings.to_csv(timings_path, index=False)

summary = {
    'run_mode': RUN_MODE,
    'pdf_count': int(len(benchmark_pdf_paths)),
    'page_count': int(len(predictions)),
    'training_seconds': float(training_seconds),
    'processing_seconds': float(processing_seconds),
    'seconds_per_pdf': float(processing_seconds / len(benchmark_pdf_paths)),
    'seconds_per_page': float(processing_seconds / len(predictions)),
    'predicted_text_pages': int((predictions.predicted_page_type == 'text').sum()),
    'predicted_drawing_pages': int((predictions.predicted_page_type == 'drawing').sum()),
}
summary_path = OUTPUT_DIR / f'{RUN_MODE}_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(json.dumps(summary, indent=2))
print('Saved predictions:', predictions_path)
print('Saved timings:', timings_path)
print('Saved summary:', summary_path)

## Interpretation

- `training_seconds` measures only the small reconstruction of V2 from saved features.
- `processing_seconds` includes PDF rendering, feature extraction, and page prediction.
- The production script can save V2 as a joblib bundle, after which normal prediction does not perform training.